## ipywidgets

**Avant de run :** dans `data_science_project` exécuter `uv sync`.

**Note :** l'outil lit seulement les parties déjà téléchargées (`uv run download-nhl`), il n'appelle jamais l'API.

In [ ]:
# Setup
import sys, ift6758
print(sys.executable)
print(ift6758.__file__)

import json
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

# Racine du projet = le dossier data_science_project (ce notebook est dans son sous-dossier notebooks/).
# Sert à trouver le cache de données (data/raw) et l'image de la patinoire (figures/nhl_rink.png).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "data_science_project")
os.environ["NHL_DATA_DIR"] = str(ROOT / "ift6758" / "data" / "data" / "raw")
print("Data directory:", os.environ["NHL_DATA_DIR"])

from ift6758.data import get_game
from ift6758.data.nhl import GAME_TYPE_CODE, get_data_dir

/Users/dieu/Cours_IFT6758/Projet_IFT6758/LNH-data-science/data_science_project/.venv/bin/python
/Users/dieu/Cours_IFT6758/Projet_IFT6758/LNH-data-science/data_science_project/ift6758/__init__.py


Data directory: /Users/dieu/Cours_IFT6758/Projet_IFT6758/LNH-data-science/data_science_project/ift6758/data/data/raw
get_data_dir(): /Users/dieu/Cours_IFT6758/Projet_IFT6758/LNH-data-science/data_science_project/ift6758/data/data/raw
2017 exists: True
JSON count 2017: 1355


In [2]:
# Widget pour choisir la saison
season_widget = widgets.Dropdown(
    options=[i for i in range(2016,2024)],
    value = 2017,
    description='Season start:',
    disabled=False,
)

# Widget pour choisir le game_id d'une partie en saison régulière
# Maximum s'ajuste automatiquement selon la saison choisie (ex: 2016-2017 = 1271 parties)
game_widget = widgets.IntSlider(
    description = "Game #",
    value = 1,
    min = 1,
    step = 1,
    max = 1,
    continuous_update=False
)

# Widget pour choisir le numéro d'événement
# Maximum s'ajuste automatiquement selon la partie choisie (ex: 2016-2017, game_id=2016020001 = 174 événements)
event_id_widget = widgets.IntSlider(
    description = "Event #",
    value = 0,
    min = 0,
    step = 1,
    max = 0,
    continuous_update=False
)

# Widget pour choisir toggle playoffs ou non
playoffs_widget = widgets.Checkbox(
    value=False,
    description='Playoffs',
    disabled=False,
    indent=False
)

def list_game_ids(season, playoffs):
    """Retourne les identifiants des parties déjà téléchargées pour une saison."""
    # Détermine le type de partie : saison régulière ou séries éliminatoires.
    game_type = "playoffs" if playoffs else "regular"

    # Construit le chemin du dossier contenant les parties de la saison.
    folder = get_data_dir() / str(season)
    if not folder.exists():
        return []

    # Recherche les fichiers JSON correspondant au type de partie sélectionné.
    # Le nom du fichier sans son extension correspond à l'identifiant du match.
    # Le tri permet de conserver l'ordre des identifiants.
    return sorted(
        p.stem
        for p in folder.glob(f"{season}{GAME_TYPE_CODE[game_type]}*.json")
    )

# Charge l'image de la patinoire utilisée pour afficher les événements.
rink_path = ROOT / "figures" / "nhl_rink.png"
if not rink_path.exists():
    raise FileNotFoundError(
        f"Image de patinoire introuvable : {rink_path}\n"
        "Vérifiez que figures/nhl_rink.png existe à la racine du projet."
    )
RINK = plt.imread(rink_path)

# Définit les limites de l'image en pieds :
# longueur de 200 pieds et largeur de 85 pieds.
RINK_EXTENT = [-100, 100, -42.5, 42.5]
print("Image de patinoire chargée :", rink_path)


def inspect_event(season, playoffs, game_number, event_number):
    """Affiche les informations d'une partie et visualise un événement sur la patinoire."""

    # Récupère les identifiants des parties téléchargées pour la sélection actuelle.
    ids = list_game_ids(season, playoffs)

    # Vérifie qu'au moins une partie a été téléchargée pour cette sélection.
    if not ids:
        print(
            f"Aucune partie téléchargée pour cette sélection dans {get_data_dir()}."
        )
        print(
            "Téléchargez la saison avec : "
            "uv run download-nhl --first <année> --last <année>"
        )
        return

    # Sélectionne une partie à partir de son numéro, qui commence à 1.
    # Limite l'indice pour éviter de dépasser le nombre de parties disponibles.
    game = get_game(ids[min(game_number, len(ids)) - 1])

    # Récupère la liste des événements enregistrés pour cette partie.
    global plays
    plays = game.get("plays") or []

    # Vérifie que la partie contient des événements.
    if not plays:
        print("Cette partie ne contient aucun événement.")
        return

    # Sélectionne l'événement à partir de son indice, qui commence à 0.
    # Si l'indice est trop grand, sélectionner le dernier événement disponible.
    play = plays[min(event_number, len(plays) - 1)]

    # Récupère les détails de l'événement (coordonnées, etc.).
    details = play.get("details") or {}
    x = details.get("xCoord")
    y = details.get("yCoord")
    etype = play.get("typeDescKey", "?")
    period = play.get("periodDescriptor", {}).get("number", "?")
    t = play.get("timeInPeriod", "?")
    eid = play.get("eventId", "?")

    # Affiche les informations principales de la partie sélectionnée.
    print("Date:", game.get("gameDate"))
    away = game.get("awayTeam", {}).get("abbrev", "?")
    home = game.get("homeTeam", {}).get("abbrev", "?")
    current_game_id = game.get("id")
    print("Match:", away, "at", home)
    print("Game ID:", current_game_id)
    print("Event ID:", eid)

    # Créer la figure et les axes pour afficher la patinoire.
    fig, ax = plt.subplots(figsize=(10, 4.5))

    # Afficher l'image de la patinoire avec les coordonnées réelles en pieds.
    ax.imshow(RINK, extent=RINK_EXTENT)

    # Positionner un marqueur SEULEMENT si les coordonnées existent.
    # scatter([None], [None]) casse souvent le plot silencieusement.
    if x is not None and y is not None:
        ax.scatter([x], [y], s=150, c="tab:green", edgecolors="black", zorder=3)
        print(f"Coordonnées : x={x}, y={y}")
    else:
        print("")
        print("Pas de coordonnées pour cet événement (pas de point vert).")
        print("Changez Event # pour un shot-on-goal, goal, hit, etc.")

    # Définir les limites des axes pour correspondre à la patinoire.
    ax.set_xlim(-100, 100)
    ax.set_ylim(-42.5, 42.5)

    # Ajouter les étiquettes des axes.
    ax.set_xlabel("x (ft)")
    ax.set_ylabel("y (ft)")

    # Ajouter un titre qui décrit l'événement
    ax.set_title(
    f"Événement : {etype.capitalize()} | Période {period} | Minute {t}",
    fontsize=14,
    )

    # Afficher la figure dans le notebook (et fermer pour éviter les fuites mémoire).
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(f"Play-by-play officiel: https://www.nhl.com/gamecenter/njd-vs-tbl/2018/04/21/{current_game_id}/playbyplay")

Image de patinoire chargée : /Users/dieu/Cours_IFT6758/Projet_IFT6758/LNH-data-science/data_science_project/figures/nhl_rink.png


In [3]:
def update_game_range(change=None):
    """Saison ou playoffs changé : ajuster le nombre de parties et revenir à la première."""
    n_games = len(list_game_ids(season_widget.value, playoffs_widget.value))
    game_widget.max = max(n_games, 1)
    game_widget.value = 1
    update_event_range()


def update_event_range(change=None):
    """Partie changée : ajuster le nombre d'événements et revenir au premier."""
    ids = list_game_ids(season_widget.value, playoffs_widget.value)
    if ids:
        n_events = len(get_game(ids[game_widget.value - 1]).get("plays") or [])
    else:
        n_events = 0
    event_id_widget.max = max(n_events - 1, 0)
    event_id_widget.value = 0


# Observateurs enregistrés AVANT interactive_output : les sliders sont ajustés avant le dessin.
season_widget.observe(update_game_range, names="value")
playoffs_widget.observe(update_game_range, names="value")
game_widget.observe(update_event_range, names="value")

update_game_range()  # initialiser les sliders avant le premier dessin

# Un seul interactive_output relié aux 4 widgets : il redessine dès que l'un d'eux change
out = widgets.interactive_output(
    inspect_event,
    {
        "season": season_widget,
        "playoffs": playoffs_widget,
        "game_number": game_widget,
        "event_number": event_id_widget,
    },
)

display(
    widgets.VBox(
        [
            widgets.HBox([season_widget, playoffs_widget]),
            game_widget,
            event_id_widget,
            out,
        ]
    )
)